# Evaluation: does simplification lose safety-critical facts?

Run `python evaluate.py` first. It writes `results/eval.csv` and `results/eval_details.json`, which this notebook reads.

**Questions we answer**
1. How much easier to read is the simplified text?
2. How often does the LLM drop or invent a critical fact on its **first try**?
3. How much does the **verify → fix → retry** loop help?
4. What kinds of facts get lost?

> ⚠️ Not medical advice. Research prototype.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../results/eval.csv")
details = json.load(open("../results/eval_details.json", encoding="utf-8"))
print(f"{df.drug.nunique()} drugs, {len(df)} label sections, model: {df.model.iloc[0]}")
df.head()

## 1. Headline results

**Fact recall** = share of critical facts (doses, times, ages, warnings) from the original that are still in the simplified text. This is the most important number. Anything below 100% means information that could matter to a patient was lost.

We compare the LLM's **first try** with the **final** output after the verify-and-retry loop.

In [ ]:
summary = pd.DataFrame({
    "First try": {
        "Fact recall (micro)": (df.first_try_recall * df.facts_total).sum() / df.facts_total.sum(),
        "Sections with 100% recall": (df.first_try_recall == 1).mean(),
        "Invented numbers (total)": df.first_try_invented.sum(),
    },
    "After verify + retry": {
        "Fact recall (micro)": df.facts_kept.sum() / df.facts_total.sum(),
        "Sections with 100% recall": (df.fact_recall == 1).mean(),
        "Invented numbers (total)": df.invented_facts.sum(),
    },
})
print(f"Sections that needed at least one retry: {(df.retries > 0).mean():.0%}")
print(f"Sections passing all hard checks at the end: {df.passed.mean():.0%}")
summary.style.format("{:.1%}", subset=pd.IndexSlice[["Fact recall (micro)", "Sections with 100% recall"], :])

## 2. Readability before vs after

**Caution:** openFDA strips punctuation from bulleted lists, so many original sections look like one giant sentence. That inflates the original's Flesch-Kincaid grade and exaggerates our improvement. The **hard-words %** metric does not depend on sentence boundaries, so it is the fairer comparison. Report both, and say so in your write-up.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (before, after, title) in zip(axes, [
    ("fk_before", "fk_after", "Flesch-Kincaid grade (inflated for originals)"),
    ("hard_words_before", "hard_words_after", "Hard words (%)"),
]):
    ax.boxplot([df[before], df[after]], tick_labels=["Original", "Simplified"])
    ax.set_title(title)
axes[0].axhline(5, ls="--", c="gray"); axes[0].text(2.3, 5, "target", va="center", c="gray")
plt.tight_layout(); plt.show()

df[["fk_before", "fk_after", "hard_words_before", "hard_words_after", "words_before", "words_after"]].describe().round(1)

## 3. Fact recall by drug

In [ ]:
by_drug = df.groupby("drug").apply(
    lambda g: pd.Series({
        "first_try": (g.first_try_recall * g.facts_total).sum() / max(g.facts_total.sum(), 1),
        "final": g.facts_kept.sum() / max(g.facts_total.sum(), 1),
    }), include_groups=False,
).sort_values("final")
by_drug.plot.barh(figsize=(8, 0.4 * len(by_drug) + 1), xlim=(0, 1.05), title="Critical-fact recall by drug")
plt.xlabel("recall"); plt.tight_layout(); plt.show()

## 4. Error analysis: what gets lost or invented?

This is where the interesting findings are. Read these examples and look for patterns: are ages dropped more than doses? Do unit conversions (24 hours → 1 day) cause false alarms?

In [ ]:
rows = []
for d in details:
    for f in d["missing"]:
        rows.append({"drug": d["drug"], "section": d["section"], "problem": "missing", "kind": f["kind"], "fact": f["text"]})
    for f in d["invented"]:
        rows.append({"drug": d["drug"], "section": d["section"], "problem": "invented", "kind": f["kind"], "fact": f["text"]})
errors = pd.DataFrame(rows)
if errors.empty:
    print("No missing or invented facts in the final outputs.")
else:
    display(errors.groupby(["problem", "kind"]).size().rename("count").to_frame())
    display(errors)

## 5. NLI meaning-check flags (for human review)

In [ ]:
flags = pd.DataFrame([
    {"drug": d["drug"], "section": d["section"], **f} for d in details for f in d["nli_flags"]
])
if flags.empty:
    print("No NLI flags (or NLI was turned off).")
else:
    display(flags.groupby(["direction", "kind"]).size().rename("count").to_frame())
    pd.set_option("display.max_colwidth", 120)
    display(flags[["drug", "section", "direction", "kind", "sentence", "score"]].head(20))

## 6. Look at one example side by side

In [ ]:
example = details[0]   # change the index to browse
print(example["drug"], "/", example["section"], "
")
print("ORIGINAL:
", example["original"], "
")
print("SIMPLIFIED:
", example["simplified"])

## Next steps for the write-up

- **Human check of the checker.** Hand-label ~50 sections: did the fact checker and NLI flags catch every real error? That gives you precision and recall *of the verifier itself*, which is a strong result.
- **SARI.** SARI compares output against human-written simplifications. There are none for drug labels, so write simple references for ~20 sections yourself and compute SARI with the Hugging Face `evaluate` library.
- **Compare models.** Change `OPENROUTER_MODEL` in `.env`, re-run `evaluate.py`, and compare first-try recall across models.
- **Ablation.** How much does the retry loop help? (Compare the *First try* and *After verify + retry* columns above.)